# 01 - Checks
Do the models and all methods work, and how long do they take?

Needs `ou_guidance/` and `resnet_cifar10.pt` in `My Drive/tfg_ou/`. Runtime: T4 GPU.
If the install cell asks to restart, restart and run from the top.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/tfg_ou'      # code, checkpoint and results

In [ ]:
# pinned: a newer diffusers clashes with transformers
!pip install -q diffusers==0.35.2 transformers==4.55.2 "huggingface-hub>=0.34,<1.0" torch-fidelity

In [ ]:
import sys
sys.path.insert(0, f'{ROOT}/ou_guidance')
import helpers
helpers.ROOT = ROOT
from helpers import run, search, show_results, diagnose

## 1. Classifiers
Both should be about 0.95 on real test images.

In [ ]:
!python {ROOT}/ou_guidance/check_models.py {ROOT}/resnet_cifar10.pt

## 2. Every method, 32 images
Saved in `results_checks`, apart from the real results.

In [ ]:
OUT = 'results_checks'
run('none',    n=32, out=OUT)
run('network', n=32, out=OUT, rho=1.0)                                            # DPS
run('network', n=32, out=OUT, rho=1.0, mu=0.25, n_iter=4, gamma=0.001)            # TFG
run('ou',      n=32, out=OUT, rho=1.0)                                            # OU-DPS
run('ou',      n=32, out=OUT, rho=1.0, mu=0.25, n_iter=4, gamma=0.001)            # OU-TFG

## 3. Expected
- unconditional: about 0.1; guided runs higher (strengths untuned)
- OU-DPS: `unet_backward = 0`, faster than DPS
- FID: a number, not an error

In [ ]:
show_results(out='results_checks')

## 4. One search step (16 images)

In [ ]:
import time
for method, family in [('ou', 'simple'), ('network', 'tfg')]:
    start = time.time()
    search(method, family, n=16, max_steps=1)
    print(f'>>> one search step, {method} / {family}: {time.time() - start:.0f} s')

## 5. Budget
A full search is up to 13 (simple) or 37 (tfg) trials of `N_SEARCH` images.

In [ ]:
import glob, json
from helpers import kind
N, N_SEARCH = 100, 64
for f in sorted(glob.glob(f'{ROOT}/results_checks/*.json')):
    r = json.load(open(f)); s = r['seconds_per_image']
    tfg_style = r['n_iter'] > 0
    search = 'none' if r['method'] == 'none' else f"{(37 if tfg_style else 13) * N_SEARCH * s / 60:.0f} min"
    print(f"{kind(r):18s}  final run of {N} images: {s * N / 60:5.1f} min   full search: {search}")